# 1.1.3 — Загрузчик реальных объектов (пиклы циклических опытов + ведомость)

Самодостаточный ноутбук: **все методы описаны внутри**. Он читает выданные объекты
испытаний разжижения, извлекает из них характеристики грунта и массивы опытов и собирает
**артефакт обучения** в формате проекта — без синтетических конструктов.

## Откуда берутся данные

Каждый объект — это папка (имя папки = имя объекта). Внутри:

| Файл | Что это |
|---|---|
| `data CyclicModel - 1.pickle` | массивы циклических опытов: для каждого образца — `cycles`, `PPR`, `strain`, `deviator`, `mean_effective_stress`, … |
| `handler CyclicModel - 1.pickle` | параметры и результаты опыта: `sigma_1, t` (→ CSR), `cycles_count`, `n_fail`/`fail_cycle` (→ N_liq), `frequency`, **`physical`** — полный паспорт грунта (e, Ip, Il, Ir, плотности, грансостав, `type_ground`, глубина …) |
| `<объект>.xls` | ведомость: тот же паспорт грунта в табличном виде (кросс-проверка) |

Объекты сгруппированы по типу воздействия — `Потенциал разжижения` и `Штормовое разжижение`.

## Ключевая идея распаковки

Пиклы сериализованы классом `CyclicData` из проекта *digitrock*. Чтобы их прочитать,
**не нужен весь digitrock** (PyQt, БД и т.п.): мы подставляем лёгкую заглушку через
кастомный `Unpickler`. Числовые массивы (numpy) распаковываются штатно.

## Контракт наблюдаемых данных (вход обучения)

Извлекаем только то, что реально доступно из опыта и паспорта:
`soil_df` (свойства грунта) · `load_df` (нагружение) · `cycles`/`csr`/`r_measured` (PPR(N)) ·
`valid_mask` · `liq_label` · `n_liq`. Параметры кривой CRR (α, β) считаются физической
моделью из свойств — это **входные признаки**. Скрытые состояния (z, истинная CRR,
непрерывный риск) не используются.

## Шаг 0. Окружение и пути к данным

In [2]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

import os, glob, pickle, warnings
warnings.filterwarnings("ignore")
from liquefaction_ai import get_default_config, save_population_artifact, set_global_seed
from liquefaction_ai.data import build_population_from_experiments, prepare_benchmark_dataset
from liquefaction_ai.physics.g0 import g0_mpa, vs_from_g0

config = get_default_config()
set_global_seed(config.seed)
SEQ_LEN = config.seq_len            # длина сетки PPR(N), узлов
print("seq_len:", SEQ_LEN, "| prefix_len:", config.prefix_len)

# --- ГДЕ ЛЕЖАТ ОБЪЕКТЫ (отредактируйте под свою машину) ---
# Корень с подпапками "Потенциал разжижения" и "Штормовое разжижение".
ROOT_CANDIDATES = [
    Path("/Users/nikita/Desktop/Новая папка 4/Облако разжижения"),
]
_env_root = os.environ.get("LIQ_REAL_ROOT", "").strip()
if _env_root:
    ROOT_CANDIDATES.insert(0, Path(_env_root))
CLOUD_ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), ROOT_CANDIDATES[-1])

# Какие типы воздействия обрабатывать (имена подпапок)
TEST_TYPES = [t for t in os.environ.get(
    "LIQ_REAL_TYPES", "Потенциал разжижения,Штормовое разжижение").split(",") if t.strip()]
# Ограничение числа объектов на тип (0 = все). Полезно для быстрой проверки:
# штормовые пиклы крупные (сотни МБ) и читаются дольше.
MAX_OBJECTS = int(os.environ.get("LIQ_REAL_MAXOBJ", "0")) or None

# Куда сохранить артефакт. Чтобы пустить весь пайплайн на реальных данных — см. «Итог».
OUTPUT_DIR = REPO_ROOT / "data" / "real_objects"
print("Корень данных:", CLOUD_ROOT, "| существует:", CLOUD_ROOT.exists())
print("Типы:", TEST_TYPES, "| лимит объектов/тип:", MAX_OBJECTS)
print("Выход:", OUTPUT_DIR)

seq_len: 72 | prefix_len: 12
Корень данных: /sessions/determined-cool-fermat/mnt/Облако разжижения | существует: True
Типы: ['Потенциал разжижения', 'Штормовое разжижение'] | лимит объектов/тип: None
Выход: /sessions/determined-cool-fermat/mnt/liquefaction-ai/data/real_objects


## Шаг 1. Безопасная распаковка пиклов

`RealUnpickler` подменяет любой класс из *digitrock* (например `CyclicData`,
`ModelTriaxialCyclicLoadingSoilTest`, обёртки результатов) лёгкой заглушкой `_Stub`,
которая принимает состояние объекта в свой `__dict__`. Массивы numpy грузятся штатно.
Так пикл читается без установки digitrock и его зависимостей.

`gv(obj, name)` — доступ к атрибуту, в т.ч. распаковка обёрток-дескрипторов результата
(где значение лежит в `_value`/`value`).

In [3]:
class _Stub:
    """Заглушка под любой неизвестный класс из digitrock: принимает state в __dict__."""
    def __setstate__(self, state):
        if isinstance(state, dict):
            self.__dict__.update(state)

class RealUnpickler(pickle.Unpickler):
    """Unpickler, не требующий установленного digitrock."""
    def find_class(self, module, name):
        if module.startswith("numpy") or module in ("builtins", "collections", "_codecs", "copy_reg"):
            return super().find_class(module, name)
        return type(name, (_Stub,), {})

def load_pickle(path):
    """Прочитать пикл data/handler без зависимостей digitrock."""
    with open(path, "rb") as f:
        return RealUnpickler(f).load()

def gv(obj, name, default=None):
    """Достать атрибут объекта; если это обёртка результата — вернуть её _value/value."""
    d = getattr(obj, "__dict__", {})
    if name not in d:
        return default
    v = d[name]
    if hasattr(v, "__dict__"):
        vd = v.__dict__
        if "_value" in vd:
            return vd["_value"]
        if "value" in vd:
            return vd["value"]
    return v

print("Методы распаковки готовы.")

Методы распаковки готовы.


## Шаг 2. Гладкая линия PPR(N) из квазисинусоиды

В сыром опыте `PPR` колеблется внутри каждого цикла (квазисинусоида). Нам нужна **гладкая
линия** по её **верхним точкам**. Делаем это в три шага (как в лаборатории + сглаживание):

1. `extract_upper_envelope` — по одному пику `PPR` на цикл (приём digitrock `define_max_rude`);
2. `monotone_smooth` — неубывающая аппроксимация пиков (изотоническая регрессия) с лёгким
   гауссовым сглаживанием: физично, ведь накопленное поровое давление не убывает;
3. `resample_to_grid` — укладка на сетку длины `seq_len` (короткий опыт — в первые узлы с
   `valid_mask`; длинный, типично шторм, — равномерный ресэмплинг).

Та же логика доступна в проекте как `liquefaction_ai.data.smooth_ppr_trajectory` —
здесь продублирована, чтобы ноутбук оставался самодостаточным.

In [4]:
def extract_upper_envelope(cyc, ppr, points_in_cycle=None):
    """Верхние точки квазисинусоиды: один пик PPR на цикл."""
    cyc = np.asarray(cyc, float); pp = np.nan_to_num(np.asarray(ppr, float))
    m = min(cyc.size, pp.size); cyc, pp = cyc[:m], pp[:m]
    n_total = max(int(np.floor(np.nanmax(cyc))) if cyc.size else 0, 1)
    if points_in_cycle and points_in_cycle >= 2 and cyc.size >= points_in_cycle:
        cp, pk = [], []
        for i in range(0, cyc.size, int(points_in_cycle)):
            w = pp[i:i + int(points_in_cycle)]
            if w.size:
                j = int(np.argmax(w)) + i; cp.append(cyc[j]); pk.append(pp[j])
        cp, pk = np.array(cp), np.array(pk); o = np.argsort(cp)
        return cp[o], pk[o]
    bins = np.clip(np.ceil(cyc).astype(int), 1, n_total) - 1
    peaks = np.zeros(n_total); np.maximum.at(peaks, bins, pp)
    for k in range(1, n_total):                       # пустые циклы — предыдущим пиком
        if peaks[k] == 0.0 and peaks[k-1] > 0.0:
            peaks[k] = peaks[k-1]
    return np.arange(1, n_total + 1, float), peaks

def monotone_smooth(peaks, smoothing=1.0, clip=(0.0, 1.05)):
    """Гладкая неубывающая линия по пикам: изотоническая регрессия + гауссово сглаживание."""
    y = np.nan_to_num(np.asarray(peaks, float))
    if y.size <= 2:
        return np.clip(np.maximum.accumulate(y), *clip)
    try:
        from sklearn.isotonic import IsotonicRegression
        iso = IsotonicRegression(y_min=clip[0], y_max=clip[1], increasing=True)\
            .fit_transform(np.arange(len(y), float), y)
    except Exception:
        iso = np.clip(np.maximum.accumulate(y), *clip)
    if smoothing and smoothing > 0:
        try:
            from scipy.ndimage import gaussian_filter1d
            iso = gaussian_filter1d(iso, sigma=float(smoothing), mode="nearest")
        except Exception:
            pass
        iso = np.maximum.accumulate(iso)              # восстановить монотонность
    return np.clip(iso, *clip)

def resample_to_grid(cycle_peaks, ppr_smooth, seq_len):
    """Уложить гладкую линию на сетку seq_len; вернуть (cycles_grid, ppr, valid_mask)."""
    L = len(ppr_smooth)
    grid = np.zeros(seq_len, np.float32); r = np.zeros(seq_len, np.float32); mask = np.zeros(seq_len, np.float32)
    if L == 0:
        return grid, r, mask
    if L <= seq_len:
        grid[:L] = cycle_peaks; r[:L] = ppr_smooth; mask[:L] = 1.0; grid[L:] = cycle_peaks[-1]
    else:
        idx = np.linspace(0, L - 1, seq_len)
        grid = np.interp(idx, np.arange(L), cycle_peaks).astype(np.float32)
        r = np.interp(idx, np.arange(L), ppr_smooth).astype(np.float32); mask[:] = 1.0
    return grid, r, mask

def smooth_ppr_trajectory(cyc, ppr, seq_len, points_in_cycle=None, smoothing=1.0):
    """Сырая квазисинусоида PPR → гладкая линия на сетке seq_len + маска и пики."""
    cp, pk = extract_upper_envelope(cyc, ppr, points_in_cycle)
    sm = monotone_smooth(pk, smoothing=smoothing)
    grid, r, mask = resample_to_grid(cp, sm, seq_len)
    return grid, r, mask, pk

print("Методы гладкой линии PPR(N) готовы.")

Методы гладкой линии PPR(N) готовы.


## Шаг 3. Грансостав → содержание мелких/глинистых частиц, Cu

Паспорт хранит 11 фракций грансостава (`granulometric_10 … granulometric_0000`). Из них
считаем долю мелких частиц (≤ 0.05 мм), глинистую фракцию (< 0.002 мм) и коэффициент
неоднородности `Cu = D60/D10`. Если грансостав не задан (типично для глин/суглинков), —
оцениваем по типу грунта ГОСТ и числу пластичности `Ip`. Здесь же — справочники
проницаемости и оценки по типу грунта.

In [5]:
GRAN = ["10","5","2","1","05","025","01","005","001","0002","0000"]
GRAN_SIZE = [10,5,2,1,0.5,0.25,0.1,0.05,0.01,0.002,0.001]   # характерный размер фракции, мм
# Проницаемость по типу грунта ГОСТ (1…9), м/с — порядковые оценки
PERM_BY_TYPE  = {1:1e-4, 2:5e-5, 3:1e-5, 4:5e-6, 5:1e-6, 6:1e-7, 7:1e-8, 8:1e-9, 9:1e-7}
# Доля мелких частиц и глинистой фракции по типу (если нет грансостава), %
FINES_BY_TYPE = {1:3, 2:4, 3:5, 4:8, 5:20, 6:45, 7:75, 8:90, 9:60}
CLAY_BY_TYPE  = {1:1, 2:1, 3:2, 4:2, 5:4, 6:8, 7:20, 8:40, 9:15}

def fines_clay_cu(phys, type_ground, Ip):
    """Вернуть (fines_content %, clay_fraction %, Cu) из грансостава или оценкой по типу."""
    gd = {k: gv(phys, f"granulometric_{k}", None) for k in GRAN}
    if any(v not in (None, "") for v in gd.values()):
        v = {k: (float(gd[k]) if gd[k] not in (None, "") else 0.0) for k in GRAN}
        fines = v["005"] + v["001"] + v["0002"] + v["0000"]   # ≤ 0.05 мм
        clay = v["0000"]                                        # < 0.002 мм
        frac_pass = np.clip(np.cumsum([v[k] for k in GRAN][::-1])[::-1], 0, 100)  # % прошедших
        d = np.array(GRAN_SIZE, float); order = np.argsort(frac_pass)
        try:
            D10 = np.interp(10, frac_pass[order], d[order])
            D60 = np.interp(60, frac_pass[order], d[order])
            Cu = float(np.clip(D60 / max(D10, 1e-4), 1.0, 200.0))
        except Exception:
            Cu = 5.0
        return float(fines), float(clay), Cu
    clay = float(np.clip(Ip * 1.4, 2, 60)) if Ip else CLAY_BY_TYPE.get(int(type_ground), 5)
    return float(FINES_BY_TYPE.get(int(type_ground), 30)), float(clay), 5.0

def dr_proxy(e):
    """Прокси относительной плотности из коэффициента пористости (e≈0.40 плотный → 1.10 рыхлый)."""
    if e is None:
        return 0.5
    return float(np.clip((1.10 - float(e)) / (1.10 - 0.40), 0.0, 1.0))

print("Методы грансостава готовы.")

Методы грансостава готовы.


## Шаг 4. Извлечение одного образца → строки `soil_df`/`load_df` + массивы

Сводим паспорт (`physical`) и параметры опыта в наблюдаемый контракт:

* **CSR** = `t / sigma_1` (как в исходной аналитике лаборатории);
* **N_liq** = цикл разрушения `fail_cycle`/`n_fail`; если разжижения нет — число
  приложенных циклов;
* **разжижение** = пик `PPR ≥ 0.95` либо есть цикл разрушения;
* **V_s** = `√(E0/ρ)` (динамический модуль и плотность из паспорта);
* недостающие свойства — оценкой по типу грунта (см. шаг 3).

In [6]:
TYPE_TO_MODE = {"Потенциал разжижения": "seismic",
                "Сейсморазжижение": "seismic",
                "Штормовое разжижение": "storm"}

def extract_test(data_obj, handler_obj, test_type, seq_len):
    """Извлечь из одного образца строки свойств/нагрузки и массивы PPR(N)."""
    cyc = gv(data_obj, "cycles"); ppr = gv(data_obj, "PPR")
    if cyc is None or ppr is None or len(cyc) < 3:
        return None
    tp = gv(handler_obj, "_test_params")
    phys = gv(tp, "physical") if tp is not None else None
    tr = gv(handler_obj, "_test_result")
    if tp is None or phys is None:
        return None

    # --- гладкая линия PPR(N) по верхней огибающей квазисинусоиды ---
    pic = gv(tp, "points_in_cycle", None)
    grid, r, mask, peaks = smooth_ppr_trajectory(cyc, ppr, seq_len, points_in_cycle=pic)
    peak = float(peaks.max()) if len(peaks) else float(np.nanmax(np.nan_to_num(ppr)))

    # --- нагрузка ---
    sigma_1 = gv(tp, "sigma_1", 100.0) or 100.0
    t = gv(tp, "t", None)
    csr = float(t) / float(sigma_1) if t else 0.2
    n_total = max(int(np.floor(np.nanmax(cyc))), 1)
    cycles_count = gv(tp, "cycles_count", n_total) or n_total
    n_max = float(max(cycles_count, n_total))

    # --- разжижение / N_liq ---
    fail = gv(tr, "fail_cycle", None) if tr is not None else None
    if fail in (None, 0):
        fail = gv(tp, "n_fail", None)
    liq = 1 if (peak >= 0.95 or fail not in (None, 0)) else 0
    n_liq = float(fail) if (liq and fail not in (None, 0)) else float(n_total)

    # --- свойства грунта ---
    tg = int(gv(phys, "type_ground", 7) or 7)
    Ip = gv(phys, "Ip", None); e = gv(phys, "e", None)
    rho = gv(phys, "r", 2.0) or 2.0
    # Vs из МАЛОДЕФОРМАЦИОННОГО модуля G0 по формулам digitrock (define_G0_threshold_shear_strain):
    # G0 = f(p_ref, e, c, φ, K0, тип грунта, Ip) из СТАТИЧЕСКИХ свойств — определён для всех
    # образцов, без выдуманных констант. Vs = sqrt(G0·1000/r) (как transverse_waves_velocity
    # в резонансной колонке digitrock). p_ref — среднее эффективное напряжение обжатия, МПа.
    sigma_3 = gv(tp, "sigma_3", None)
    s3 = float(sigma_3) if sigma_3 else float(sigma_1)
    p_ref = (float(sigma_1) + 2.0 * s3) / 3.0 / 1000.0          # кПа → МПа
    c_par = gv(tp, "c", None); fi_par = gv(tp, "fi", None); K0_par = gv(tp, "K0", 0.5) or 0.5
    G0 = float(g0_mpa([p_ref], [e if e else 0.65], [c_par], [fi_par], [K0_par], [tg], [Ip])[0])
    Vs = float(vs_from_g0([G0], [rho])[0])
    # измерен ли динамический модуль в опыте (диагностика; в Vs не участвует)
    Vs_measured = 1 if (gv(tr, "transverse_waves_velocity", None) or gv(tr, "G", None) or gv(tr, "E0", None)) else 0
    sigma_eff = float(sigma_1)
    fines, clay, Cu = fines_clay_cu(phys, tg, Ip)

    soil = dict(
        laboratory_number=gv(phys, "laboratory_number", ""), borehole=gv(phys, "borehole", ""),
        depth=gv(phys, "depth", None), soil_name=gv(phys, "soil_name", ""),
        type_ground=tg, class_id=tg - 1,
        e=float(e) if e is not None else 0.7, D_r=dr_proxy(e),
        I_p=float(Ip) if Ip is not None else 0.0,
        Il=float(gv(phys, "Il", 0.0) or 0.0), Ir=float(gv(phys, "Ir", 0.0) or 0.0),
        G0=G0, V_s=Vs,
        Vs1=(float(np.clip(Vs * (100.0 / max(sigma_eff, 1.0)) ** 0.25, 60, 450))
             if np.isfinite(Vs) else np.nan),
        Vs_measured=int(Vs_measured),
        xi=float((gv(tp, "damping_ratio", 1.5) or 1.5) / 100.0),
        sigma_eff=sigma_eff, permeability=PERM_BY_TYPE.get(tg, 1e-7),
        fines_content=fines, clay_fraction=clay, Cu=Cu,
        K0=float(gv(tp, "K0", 0.5) or 0.5), static_shear_ratio=0.0,
        ige=str(gv(phys, "ige", "") or ""),   # ИГЭ — для группировки кривых CRR
    )
    # Сырые поля паспорта/опыта (для богатого анализа; недостающие → NaN/0)
    def _f(x):
        try: return float(x)
        except (TypeError, ValueError): return np.nan
    soil.update(dict(
        rs=_f(gv(phys, "rs", None)), rd=_f(gv(phys, "rd", None)), r=_f(rho),
        n_porosity=_f(gv(phys, "n", None)), W=_f(gv(phys, "W", None)),
        Wl=_f(gv(phys, "Wl", None)), Wp=_f(gv(phys, "Wp", None)),
        saturation=_f(gv(phys, "Sr", None)), ground_water_depth=_f(gv(phys, "ground_water_depth", None)),
        B_value=_f(gv(phys, "skempton_initial", None)), cohesion=_f(gv(tp, "c", None)),
        phi=_f(gv(tp, "fi", None)), E_modulus=_f(gv(tp, "E", None)),
        damping_ratio=_f(gv(tp, "damping_ratio", None)),
        calcite=_f(gv(phys, "calcite", None)), dolomite=_f(gv(phys, "dolomite", None)),
    ))
    for col, key in zip(["gran_10","gran_5","gran_2","gran_1","gran_05","gran_025","gran_01","gran_005","gran_001","gran_0002","gran_0000"],
                        GRAN):
        g = gv(phys, f"granulometric_{key}", None)
        soil[col] = 0.0 if g in (None, "") else float(g)
    load = dict(
        CSR_base=csr, frequency=float(gv(tp, "frequency", 0.5) or 0.5),
        amp_scale=1.0, N_max=n_max,
        nonstationarity=0.30 if test_type == "Штормовое разжижение" else 0.05,
        load_mode=TYPE_TO_MODE.get(test_type, "seismic"),
    )
    arrays = dict(
        cycles=grid.astype(np.float32), csr=np.full(seq_len, csr, np.float32),
        r=r.astype(np.float32), mask=mask.astype(np.float32),
    )
    return soil, load, arrays, int(liq), float(n_liq)

print("Метод извлечения образца готов.")

Метод извлечения образца готов.


## Шаг 5. Загрузка объекта и обход всех объектов

`load_object` находит в папке объекта пару пиклов (`data`/`handler`), сопоставляет образцы
по лабораторному номеру и извлекает их. `discover_objects` перечисляет объекты внутри типа
воздействия. Объекты обрабатываются по одному — память освобождается (штормовые пиклы
крупные).

In [7]:
def _find_pickles(obj_dir):
    dp = glob.glob(os.path.join(obj_dir, "**", "data CyclicModel*"), recursive=True)
    hp = glob.glob(os.path.join(obj_dir, "**", "handler CyclicModel*"), recursive=True)
    return (dp[0], hp[0]) if dp and hp else (None, None)

def load_object(obj_dir, test_type, seq_len):
    """Извлечь все образцы одного объекта → список записей."""
    dpath, hpath = _find_pickles(obj_dir)
    if not dpath:
        return []
    D = load_pickle(dpath)["data"]
    H = load_pickle(hpath)["data"]
    out = []
    for key in D:
        if key not in H:
            continue
        rec = extract_test(D[key], H[key], test_type, seq_len)
        if rec is not None:
            out.append((key, rec))
    return out

def discover_objects(type_dir):
    """Список (имя, путь) объектов внутри папки типа воздействия."""
    if not os.path.isdir(type_dir):
        return []
    res = []
    for name in sorted(os.listdir(type_dir)):
        p = os.path.join(type_dir, name)
        if os.path.isdir(p) and glob.glob(os.path.join(p, "**", "data CyclicModel*"), recursive=True):
            res.append((name, p))
    return res

print("Методы загрузки объектов готовы.")

Методы загрузки объектов готовы.


## Шаг 6. Сбор всех объектов в единые массивы

Проходим по типам воздействия и объектам, накапливаем строки свойств/нагрузки и массивы
PPR(N). Каждый образец помечается тегом `объект/тип` для дальнейшей диагностики.

In [8]:
soil_rows, load_rows = [], []
CY, CS, RM, VM, LB, NL, TAG, LAB = [], [], [], [], [], [], [], []

for test_type in TEST_TYPES:
    type_dir = str(CLOUD_ROOT / test_type)
    objects = discover_objects(type_dir)
    if MAX_OBJECTS:
        objects = objects[:MAX_OBJECTS]
    print(f"[{test_type}] объектов: {len(objects)}")
    for oname, opath in objects:
        recs = load_object(opath, test_type, SEQ_LEN)
        for key, (soil, load, arr, liq, nl) in recs:
            soil_rows.append(soil); load_rows.append(load)
            CY.append(arr["cycles"]); CS.append(arr["csr"]); RM.append(arr["r"]); VM.append(arr["mask"])
            LB.append(liq); NL.append(nl); TAG.append(f"{test_type}/{oname}"); LAB.append(key)
        print(f"    {oname[:48]:48s} образцов: {len(recs)}")

soil_df = pd.DataFrame(soil_rows)
load_df = pd.DataFrame(load_rows)
cycles = np.array(CY); csr = np.array(CS)
r_measured = np.array(RM); valid_mask = np.array(VM)
liq_label = np.array(LB); n_liq = np.array(NL)
soil_df["object"] = TAG

# Vs построен из G0 (формула digitrock) по статическим свойствам — определён для всех образцов.
_n_meas = int(soil_df["Vs_measured"].sum())
print(f"Vs из G0 (digitrock) для всех {len(soil_df)} образцов; динамический модуль измерен у "
      f"{_n_meas} ({100*_n_meas/len(soil_df):.1f}%) — используется только как диагностика.")
print("Vs, м/с:  min=%.0f  median=%.0f  max=%.0f" % (soil_df["V_s"].min(), soil_df["V_s"].median(), soil_df["V_s"].max()))

assert len(soil_df) > 0, "Объекты не найдены — проверьте CLOUD_ROOT/TEST_TYPES."

[Потенциал разжижения] объектов: 3
    333-24 Дубнинская - Plaxis G0                    образцов: 71


    338-24 АД Москва - Казань - мех                  образцов: 96


    852-23 3-й Хорошевский пр-д, вл. 5 (ВГК-5)- pl образцов: 96
[Штормовое разжижение] объектов: 8


    196-24 ул. Б. Татарская                          образцов: 54


    252-24 Усиление эстакады                         образцов: 25


    255-24 Б. Садовая 8 - plaxis (35)                образцов: 31


    390-24 Шаболовская -Plaxis (66)                  образцов: 66
    613-23 Тушино - plaxis (23)                      образцов: 23


    638-24 Ленинградское шоссе                       образцов: 66


    818-23 Руза гора - plaxis (48)                   образцов: 48


    856-23 3-й Хорошевский пр-д, вл. 5 (ВГК-6) - P образцов: 90
Vs из G0 (digitrock) для всех 666 образцов; динамический модуль измерен у 111 (16.7%) — используется только как диагностика.
Vs, м/с:  min=80  median=164  max=244


## Шаг 7 (опционально). Прямой парсинг ведомости `.xls`

Паспорт грунта из `handler`-пикла — это уже оцифрованная ведомость, поэтому для обучения
**достаточно пиклов**. Но при необходимости свойства можно прочитать прямо из `.xls` по
фиксированным позициям колонок (как это делает digitrock). Ниже — самостоятельный
читатель ведомости для кросс-проверки: он возвращает таблицу свойств, индексированную
лабораторным номером.

In [9]:
# Карта позиций колонок ведомости (как в digitrock: PhysicalPropertyPosition)
STATEMENT_COLS = {
    "laboratory_number":0, "borehole":1, "depth":2, "soil_name":3,
    "granulometric_10":4, "granulometric_5":5, "granulometric_2":6, "granulometric_1":7,
    "granulometric_05":8, "granulometric_025":9, "granulometric_01":10, "granulometric_005":11,
    "granulometric_001":12, "granulometric_0002":13, "granulometric_0000":14,
    "rs":15, "r":16, "rd":17, "n":18, "e":19, "W":20, "Sr":21, "Wl":22, "Wp":23,
    "Ip":24, "Il":25, "Ir":30, "ground_water_depth":35,
}

def read_statement(xls_path):
    """Прочитать ведомость .xls в таблицу свойств (индекс — лабораторный номер).

    Требует пакет xlrd (для старого формата .xls): pip install xlrd.
    Строка данных определяется автоматически — первая, где в колонке A непустой
    лабораторный номер вида '1-1'/'12'.
    """
    raw = pd.read_excel(xls_path, header=None, engine="xlrd")
    start = None
    import re
    for i in range(min(40, len(raw))):
        a = str(raw.iat[i, 0]).strip()
        if re.match(r"^\d+(-\d+)?$", a):
            start = i; break
    if start is None:
        return pd.DataFrame()
    rows = []
    for i in range(start, len(raw)):
        a = str(raw.iat[i, 0]).strip()
        if not re.match(r"^\d+(-\d+)?$", a):
            continue
        rows.append({name: raw.iat[i, col] for name, col in STATEMENT_COLS.items()})
    df = pd.DataFrame(rows).set_index("laboratory_number")
    return df

# Демонстрация на первом доступном объекте (если есть xlrd и .xls):
try:
    _type0 = TEST_TYPES[0]; _objs = discover_objects(str(CLOUD_ROOT / _type0))
    if MAX_OBJECTS: _objs = _objs[:MAX_OBJECTS]
    _xls = glob.glob(os.path.join(_objs[0][1], "*.xls")) if _objs else []
    if _xls:
        _st = read_statement(_xls[0])
        print("Ведомость:", os.path.basename(_xls[0]), "| образцов:", len(_st))
        display(_st[["depth","soil_name","e","Ip","Il","Ir"]].head(5))
    else:
        print("Файл .xls не найден — шаг пропущен (для обучения он не требуется).")
except Exception as err:
    print("Парсинг ведомости пропущен:", type(err).__name__, str(err)[:120])

Ведомость: 333-24 Дубнинская - Plaxis G0.xls | образцов: 72


,depth,soil_name,e,Ip,Il,Ir
laboratory_number,,,,,,
0,2.0,3,19.00,24.0,25.0,30.0
6-1,1.5,Песок пылеватый однородный,0.72,NaN,NaN,NaN
9-4,2.0,Песок пылеватый однородный,0.72,NaN,NaN,NaN
11-5,2.0,Песок пылеватый однородный,0.72,NaN,NaN,NaN
12-11,2.5,Песок пылеватый однородный,0.72,NaN,NaN,NaN


## Шаг 8. Кривая потенциала разжижения CRR(N) для нейросети

Для «Потенциала разжижения» образцы одного **ИГЭ** испытывают при разных CSR — каждый даёт
точку (N_fail, CSR). По группе строится кривая потенциала **CRR(N) = β/N^(1−α)** (как в
digitrock: `approximate_test_data` — фит степенной кривой по `(N_fail, CSR)`). Эта измеренная
граница подаётся в адаптер как `crr_obs` с по-образцовой маской `crr_obs_mask` — голова CRR
нейросети обучается по реальному измерению там, где серия доступна (а не только по физической
модели). Для штормовых опытов кривой потенциала нет → маска 0.

In [10]:
from scipy.optimize import curve_fit, differential_evolution

def _hyp(N, alpha, betta):
    """Кривая CSR(N)=β/N^(1−α) (define_cyclic_stress_ratio_hyp из digitrock)."""
    return betta / (N ** (1.0 - alpha))

def fit_alpha_betta(cycles, csr):
    """Фит (α, β) по точкам (N_fail, CSR), как approximate_test_data в digitrock."""
    cycles = np.asarray(cycles, float); csr = np.asarray(csr, float)
    m = (cycles > 0) & np.isfinite(cycles) & np.isfinite(csr) & (csr > 0)
    cycles, csr = cycles[m], csr[m]
    if len(cycles) < 3:
        return None
    try:
        de = differential_evolution(lambda p: float(np.sum((csr - _hyp(cycles, *p)) ** 2)),
                                    [(0, 1), (0, 1.5)], seed=3, maxiter=60, tol=1e-3, polish=False)
        popt, _ = curve_fit(_hyp, cycles, csr, p0=de.x, bounds=([0, 0], [0.999, 3.0]), maxfev=5000)
        a, b = float(popt[0]), float(popt[1])
        return (a, b) if (np.isfinite(a) and np.isfinite(b)) else None
    except Exception:
        return None

def build_crr_obs(soil_df, load_df, cycles, n_liq, liq_label, tags, seq_len, min_group=3):
    """CRR(N) по ИГЭ: фит β/N^(1−α) по (N_fail, CSR) разрушившихся образцов потенциала."""
    n = len(soil_df)
    crr = np.zeros((n, seq_len), np.float32); mask = np.zeros(n, np.float32)
    is_pot = np.array(["Потенциал" in str(t) for t in tags])
    csr_base = load_df["CSR_base"].to_numpy()
    key = (soil_df["object"].astype(str) + "|" + soil_df["ige"].astype(str)).to_numpy()
    df = pd.DataFrame({"i": np.arange(n), "key": key})
    n_groups = 0
    for _, grp in df.groupby("key"):
        idx = grp["i"].to_numpy(); pidx = idx[is_pot[idx]]
        fl = pidx[liq_label[pidx] == 1]
        if len(fl) < min_group:
            continue
        res = fit_alpha_betta(n_liq[fl], csr_base[fl])
        if res is None:
            continue
        a, b = res; n_groups += 1
        for i in pidx:
            crr[i] = np.clip(b / np.maximum(cycles[i], 1.0) ** (1.0 - a), 0.02, 0.9).astype(np.float32)
            mask[i] = 1.0
    return crr, mask, n_groups

crr_obs, crr_obs_mask, n_groups = build_crr_obs(soil_df, load_df, cycles, n_liq, liq_label,
                                                soil_df["object"].tolist(), SEQ_LEN)
print(f"Кривых CRR(N) построено (групп ИГЭ): {n_groups} | образцов с измеренной CRR: "
      f"{int(crr_obs_mask.sum())} из {int(crr_obs_mask.shape[0])}")
if crr_obs_mask.sum() == 0:
    crr_obs = crr_obs_mask = None    # нет серий потенциала → без измеренной CRR

Кривых CRR(N) построено (групп ИГЭ): 44 | образцов с измеренной CRR: 263 из 666


## Шаг 9. Сборка артефакта обучения

`build_population_from_experiments` строит признаки (включая α/β и факторы CRR из физической
модели), метаданные, наблюдаемые цели (`g_obs`, `risk_proxy`) и стратифицированное
benchmark-разбиение — в формате, идентичном синтетическому артефакту, но без латентных
полей.

In [11]:
population = build_population_from_experiments(
    soil_df=soil_df, load_df=load_df, cycles=cycles, csr=csr, r_measured=r_measured,
    valid_mask=valid_mask, liq_label=liq_label, n_liq=n_liq, config=config,
    crr_obs=crr_obs, crr_obs_mask=crr_obs_mask)
save_population_artifact(OUTPUT_DIR, population, config)

has_latent = any(k in population for k in ["z_true", "g_true", "crr_mix"])
print("Сохранено в:", OUTPUT_DIR)
print("static_dim:", population["static_features"].shape[1],
      "| seq:", population["seq_inputs"].shape,
      "| латентные синтетические поля:", has_latent)
print("колонок meta:", population["meta"].shape[1],
      "| измеренная CRR:", "есть" if "crr_obs" in population else "нет")

Сохранено в: /sessions/determined-cool-fermat/mnt/liquefaction-ai/data/real_objects
static_dim: 34 | seq: (666, 72, 5) | латентные синтетические поля: False
колонок meta: 98 | измеренная CRR: есть


## Шаг 10. Проверка: датасет готов к обучению

In [12]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark = prepare_benchmark_dataset(population, config, device)
for split_name in ["train", "val", "test"]:
    s = benchmark[split_name]
    print(f"{split_name:6s}: static={tuple(s['static'].shape)}, seq={tuple(s['seq_in'].shape)}, "
          f"liq_rate={float(s['label'].float().mean()):.3f}")

meta = population["meta"]
display(pd.DataFrame({
    "Показатель": ["образцов", "доля разжижения", "средний N_liq, циклов",
                   "медианный пик PPR", "средняя CRR15", "типов грунта", "объектов"],
    "Значение": [len(meta), round(float(meta["liq_label"].mean()), 4),
                 round(float(meta["N_liq_true"].mean()), 1),
                 round(float(meta["PPR_max_true"].median()), 4),
                 round(float(meta["crr_ref"].mean()), 4),
                 int(soil_df["type_ground"].nunique()),
                 int(soil_df["object"].nunique())],
}))

train : static=(466, 34), seq=(466, 72, 5), liq_rate=0.607
val   : static=(99, 34), seq=(99, 72, 5), liq_rate=0.606
test  : static=(101, 34), seq=(101, 72, 5), liq_rate=0.614


,Показатель,Значение
0,образцов,666.0000
1,доля разжижения,0.6081
2,"средний N_liq, циклов",2429.1000
3,медианный пик PPR,1.0030
4,средняя CRR15,0.3005
5,типов грунта,8.0000
6,объектов,11.0000


## Шаг 11. Реальные гладкие линии PPR(N)

Несколько кривых порового давления после выделения верхней огибающей и монотонного
сглаживания — наглядная проверка, что в обучение пошли именно опытные данные (гладкая
линия по верхним точкам квазисинусоиды), а не синтетика.

In [13]:
import plotly.graph_objects as go
from liquefaction_ai.viz import save_figure

rng = np.random.default_rng(config.seed)
sample_idx = rng.choice(len(soil_df), size=min(12, len(soil_df)), replace=False)
fig = go.Figure()
for i in sample_idx:
    m = valid_mask[i] > 0
    fig.add_trace(go.Scatter(
        x=cycles[i][m], y=r_measured[i][m], mode="lines",
        name=f"{soil_df['laboratory_number'].iloc[i]} (CSR={load_df['CSR_base'].iloc[i]:.2f})",
        hovertemplate="N=%{x:.0f}, PPR=%{y:.2f}<extra></extra>"))
fig.add_hline(y=1.0, line_dash="dash", line_color="red",
              annotation_text="PPR=1 (разжижение)")
fig.update_layout(title="Гладкие линии PPR(N) реальных образцов (верхняя огибающая)",
                  xaxis_title="Число циклов N", yaxis_title="PPR, д.е.",
                  legend_title="Образец", height=460)
if SAVE_FIGS:
    save_figure(fig, "1_1_3_real_ppr_trajectories")
fig.show()

## Итог

Артефакт из **реальных объектов** собран и сохранён в `data/real_objects`. Он содержит только
наблюдаемые величины (измеренный PPR, метку разжижения, N_liq) и признаки CRR, вычисленные
физической моделью из свойств грунта.

**Как пустить весь пайплайн (анализ → обучение → оценка) на реальных данных:**

1. поставьте в шаге 0 `OUTPUT_DIR = DATA_DIR` (тогда основной датасет проекта будет браться
   из реальных данных), **или**
2. скопируйте каталог `data/real_objects` в `data/demo_run`.

Дальнейшие ноутбуки (1.2 анализ, 1.3 CRR-параметры, 1.4 сплиты, серии 2–3) работают поверх
этого артефакта без изменений. Обучение и оценка используют только наблюдаемые сигналы —
никаких синтетических конструктов.

**Заметки по данным.** Штормовые пиклы крупные (сотни МБ, длинные опыты до тысяч циклов) —
их чтение занимает время; используйте `LIQ_REAL_MAXOBJ` для быстрой проверки. В «Потенциале
разжижения» разжижаются почти все образцы (CSR подбирается до разрушения), поэтому ключевой
сигнал — N_liq; стратификация разбиения автоматически огрубляется при вырожденной метке.